# 06_finetune_marian_en_hi

In [1]:
from transformers import MarianMTModel, MarianTokenizer
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-hi")
model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-hi").to(device)


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [3]:
from datasets import load_from_disk
dataset = load_from_disk("../data/processed/iitb_filtered")

In [4]:
SUBSET_SIZE = 180_000  # adjust as needed

train_subset = dataset["train"].shuffle(seed=42).select(range(SUBSET_SIZE))

In [5]:
val_subset = dataset["validation"].shuffle(seed=42).select(range(min(10_000, len(dataset["validation"]))))
test_subset = dataset["test"].shuffle(seed=42).select(range(min(10_000, len(dataset["test"]))))

In [13]:
def tokenize_batch(sources, targets, tokenizer, max_length=80):
    """
    Tokenizes a batch of source (English) and target (Hindi) sentences
    for MarianMT training. Returns input_ids, attention_mask, and labels
    with padding positions in labels set to -100 (so loss ignores them).
    """
    model_inputs = tokenizer(
        sources,
        text_target=targets,
        max_length=max_length,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    # Replace pad_token_id with -100 in labels, so CrossEntropyLoss ignores padding
    labels = model_inputs["labels"]
    labels[labels == tokenizer.pad_token_id] = -100
    model_inputs["labels"] = labels

    return model_inputs

In [14]:
sample_sources = ["Hello, how are you?", "This is a test."]
sample_targets = ["नमस्ते, आप कैसे हैं?", "यह एक परीक्षण है।"]

batch = tokenize_batch(sample_sources, sample_targets, tokenizer)

print(batch["input_ids"])
print(batch["attention_mask"])
print(batch["labels"])

tensor([[12110,     2,   287,    54,    27,    22,     0, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949],
        [  239,    23,    19,  2564,     3,     0, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949, 61949,
         61949, 61949, 61949, 61949, 61949, 61949, 

In [15]:
from torch.utils.data import DataLoader, Dataset

class TranslationDataset(Dataset):
    def __init__(self, sources, targets, tokenizer, max_length=80):
        self.sources = sources
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sources)

    def __getitem__(self, idx):
        source = self.sources[idx]
        target = self.targets[idx]
        tokenized = tokenize_batch([source], [target], self.tokenizer, self.max_length)
        return {key: val.squeeze(0) for key, val in tokenized.items()}

In [16]:
train_dataset = TranslationDataset(train_subset["en"], train_subset["hi"], tokenizer)
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)

In [17]:
val_dataset = TranslationDataset(val_subset["en"], val_subset["hi"], tokenizer)
val_dataloader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [18]:
batch = next(iter(train_dataloader))
print(batch["input_ids"].shape)

torch.Size([16, 80])


In [19]:
from torch.optim import AdamW
optimizer = AdamW(model.parameters(), lr=3e-5)

In [20]:
import time

start = time.time()
for i, batch in enumerate(train_dataloader):
    if i >= 20:
        break
    batch = {k: v.to(device) for k, v in batch.items()}
    optimizer.zero_grad()
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    optimizer.step()

elapsed = time.time() - start
per_batch = elapsed / 20
total_batches = len(train_dataloader)
print(f"{per_batch:.3f}s/batch, estimated epoch time: {(per_batch * total_batches)/60:.1f} min")

0.357s/batch, estimated epoch time: 66.9 min


In [21]:
def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0

    for i, batch in enumerate(dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        if i % 50 == 0:
            print(f"  Batch {i}/{len(dataloader)} — loss: {loss.item():.4f}")

    return total_loss / len(dataloader)

In [22]:
@torch.no_grad()
def evaluate_epoch(model, dataloader, device):
    model.eval()
    total_loss = 0.0

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        total_loss += outputs.loss.item()

    return total_loss / len(dataloader)